<a href="https://colab.research.google.com/github/jaloaiza/genaiassignments/blob/main/assignments/06/Module6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generate Synthetic Training Data

<a target="_blank" href="https://colab.research.google.com/github/simonguest/CS-394/blob/main/src/06/notebooks/generate-synthetic.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://github.com/simonguest/CS-394/raw/refs/heads/main/src/06/notebooks/generate-synthetic.ipynb">
  <img src="https://img.shields.io/badge/Download_.ipynb-blue" alt="Download .ipynb"/>
</a>

## Data generation settings

In [41]:
NUM_TRAIN_EXAMPLES = 10  # @param {type:"number"}
NUM_VAL_EXAMPLES = 3  # @param {type:"number"}
NUM_TEST_EXAMPLES = 3 # @param {type:"number"}
TEMPERATURE = 0.8  # @param {type:"number"}

DATA_FOLDER = "./.data/generated"
!mkdir -p {DATA_FOLDER}

DATAGEN_URL = "https://openrouter.ai/api/v1"
DATAGEN_MODEL = "openai/gpt-5.1-chat"

## Dataset diversity

In [42]:
TOPICS_CHILD = [
    "animals_common",
    "colors_shapes",
    "weather_basic",
    "toys",
    "school_objects",
    "family_roles",
    "food_simple"
]

TOPICS_TEEN = [
    "technology_basic",
    "friendship",
    "emotions_basic",
    "sports_common",
    "school_life",
    "music_general",
    "time_concepts",
    "dreams_goals"
]

TOPICS_ADULT = [
    "work_life",
    "money",
    "relationships",
    "parenthood",
    "memory",
    "aging",
    "transportation",
    "communication",
    "habits"
]

AGE_GROUPS = ["child", "teen", "adult"]
AGE_GROUP_WEIGHTS = [0.33, 0.33, 0.34]

DIFFICULTY_LEVELS = [
    "easy",
    "medium",
    "hard",
    "trick"
]

DIFFICULTY_LEVEL_WEIGHTS = [0.4, 0.3, 0.2, 0.1]

RIDDLE_LENGTH = [
    "short",
    "medium",
    "paragraph"
]

RIDDLE_LENGTH_WEIGHTS = [0.4, 0.35, 0.25]

ANSWER_LENGTH = [
    "single_word",
    "short_phrase",
    "one_sentence"
]

ANSWER_LENGTH_WEIGHTS = [0.4, 0.35, 0.25]

ANSWER_TYPE = [
    "concrete_object",
    "abstract_concept",
    "event_or_situation"
]

ANSWER_TYPE_WEIGHTS = [0.4, 0.35, 0.25]

OUTPUT_FORMAT = [
    "riddle_answer",
    "riddle_answer_explanation",
    "riddle_only",
    "answer_only"
]

## Model for structured output

In [43]:
from pydantic import BaseModel

class RiddleExample (BaseModel):
    topic: str
    difficulty: str
    riddle_length: str
    vocabulary_level: str
    answer_length: str
    tone_style: str
    riddle: str
    answer: str
    explanation: str

## Get OpenRouter API key

In [44]:
import sys
import os
from dotenv import load_dotenv

if 'google.colab' in sys.modules:
  from google.colab import userdata # type:ignore
  os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
else:
  load_dotenv()

## Conversation generation functions

In [45]:
from httpx._transports.default import A
import openai
import os
from dataclasses import dataclass

client = openai.OpenAI(
    base_url=DATAGEN_URL,
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

def generate_completion(prompt: str) -> RiddleExample | None:
    response = client.responses.parse(
        model=DATAGEN_MODEL,
        input=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        stream=False,
        text_format=RiddleExample
    )

    return response.output_parsed

def map_riddle_length(riddle_length: str, topic: str) -> str:
    if riddle_length == "short":
        return f"a short riddle (1–2 sentences) about {topic}"
    elif riddle_length == "medium":
        return f"a medium-style riddle (3-4 lines) about {topic}"
    elif riddle_length == "paragraph":
        return f"a short story-style riddle (5–6 lines) about {topic}"
    else:
        return f"a riddle about {topic}"


def create_conversation(
    topic: str,
    riddle_length: str,
    difficulty_level: str,
    answer_length: str,
    answer_type: str,
    age_group: str,
    output_format: str) -> RiddleExample | None:

    request = map_riddle_length(riddle_length, topic)

    prompt = f"""
    Generate {request}.

    Constraints:
    - Difficulty level: {difficulty_level}
    - The riddle should tailor to this age group: {age_group}
    - The riddle should be of length: {riddle_length}

    Answer requirements:
    - The answer is of type {answer_type}
    - The answer should be {answer_length}.
    - The answer must logically match the riddle.
    - Do not reveal the answer inside the riddle.

    Return in the following format with this output format {output_format}
    and the following fields if it follows the output format:

    RIDDLE:
    <riddle text>

    ANSWER:
    <answer only>

    EXPLANATION:
    <2-4 sentence explanation of why the answer is correct>
    """

    return generate_completion(prompt)

## Dataset generation functions

In [46]:
import random
import json
from tqdm import tqdm


AGE_GROUPS = ["child", "teen", "adult"]
AGE_GROUP_WEIGHTS = [0.33, 0.33, 0.34]


def sample_topic():
    age_group = random.choices(
        AGE_GROUPS,
        weights=AGE_GROUP_WEIGHTS
    )[0]

    if age_group == "child":
        topic = random.choice(TOPICS_CHILD)
        difficulty_weights = [0.6, 0.3, 0.1, 0.0]  # easy-heavy
        answer_type_weights = [0.7, 0.2, 0.1]
    elif age_group == "teen":
        topic = random.choice(TOPICS_TEEN)
        difficulty_weights = [0.3, 0.4, 0.2, 0.1]
        answer_type_weights = [0.4, 0.4, 0.2]
    else:
        topic = random.choice(TOPICS_ADULT)
        difficulty_weights = DIFFICULTY_LEVEL_WEIGHTS
        answer_type_weights = ANSWER_TYPE_WEIGHTS

    return age_group, topic, difficulty_weights, answer_type_weights


def generate_dataset(num_examples: int, filename: str) -> None:
    with open(filename, "w", encoding="utf-8") as f:
        for idx in tqdm(range(num_examples)):
            age_group, topic, difficulty_weights, answer_type_weights = sample_topic()

            difficulty_level = random.choices(
                DIFFICULTY_LEVELS,
                weights=difficulty_weights
            )[0]

            riddle_length = random.choices(
                RIDDLE_LENGTH,
                weights=RIDDLE_LENGTH_WEIGHTS
            )[0]

            answer_length = random.choices(
                ANSWER_LENGTH,
                weights=ANSWER_LENGTH_WEIGHTS
            )[0]

            answer_type = random.choices(
                ANSWER_TYPE,
                weights=ANSWER_TYPE_WEIGHTS
            )[0]

            output_format = random.choice(OUTPUT_FORMAT)

            # Generate riddle
            conversation = None
            while conversation is None:
                conversation = create_conversation(
                    topic=topic,
                    riddle_length = riddle_length,
                    difficulty_level = difficulty_level,
                    answer_length = answer_length,
                    answer_type = answer_type,
                    age_group=age_group,
                    output_format=output_format
                )

            # Build the user prompt
            user_prompt = (
                f"Create a {difficulty_level} {riddle_length} riddle "
                f"about {topic} suitable for a {age_group} audience. "
                f"The answer should be a {answer_type} and {answer_length}. "
                f"Return format: {output_format}."
            )

            # Structured training template
            template = {
            "messages": [
                {
                    "role": "user",
                    "content": user_prompt
                },
                {
                    "role": "assistant",
                    "content": f"""RIDDLE:

                    {conversation.riddle}

                    ANSWER:
                    {conversation.answer}

                    EXPLANATION:
                    {conversation.explanation}
                    """
                }
               ]
           }

            line = json.dumps(template, ensure_ascii=False) + "\n"
            f.write(line)

        f.flush()

## Generate all the data!

In [47]:
from datetime import datetime

TRAIN_FILE = f"{DATA_FOLDER}/train_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"
VALID_FILE = f"{DATA_FOLDER}/valid_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"
TEST_FILE = f"{DATA_FOLDER}/test_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"

generate_dataset(NUM_TRAIN_EXAMPLES, TEST_FILE)


 40%|████      | 4/10 [04:31<06:46, 67.79s/it]


ValidationError: 1 validation error for RiddleExample
  Invalid JSON: EOF while parsing an object at line 4715 column 0 [type=json_invalid, input_value='{"topic":"family_roles",...\r \n \r \n \r \n \r \n', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid